In [ ]:
!python --version
!nvcc --version
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Python 3.11.11
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpvs189234".


In [ ]:
%%cuda
#include <stdio.h>
__global__ void hello(){
 printf("Hello from block: %u, thread: %u\n", blockIdx.x, threadIdx.x);
}
int main(){
 hello<<<2, 2>>>();
 cudaDeviceSynchronize();
 //en commentant cette ligne est importante pour avoir le résultat car il est le host est le responsable de la synchronisation et si on l appelle pas il nya pas de returnn
}

Hello from block: 0, thread: 0
Hello from block: 0, thread: 1
Hello from block: 1, thread: 0
Hello from block: 1, thread: 1



In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/CHPS0802/'
if not os.path.exists(path):
    print(f"Path does not exist: {path}")
else:
    print("Path exists, you can access files.")


Path exists, you can access files.


In [ ]:
%cd /content/drive/MyDrive/CHPS0802/

/content/drive/MyDrive/CHPS0802


In [ ]:
!nvcc prac1a.cu -o prac1a -I/content/drive/MyDrive/CHPS0802 -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -L /lib64 -lcudart


ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z15my_first_kernelPf' for 'sm_70'
ptxas info    : Function properties for _Z15my_first_kernelPf
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 10 registers, 360 bytes cmem[0]


In [ ]:
!./prac1a

 n,  x  =  0  0.000000 
 n,  x  =  1  1.000000 
 n,  x  =  2  2.000000 
 n,  x  =  3  3.000000 
 n,  x  =  4  4.000000 
 n,  x  =  5  5.000000 
 n,  x  =  6  6.000000 
 n,  x  =  7  7.000000 
 n,  x  =  8  0.000000 
 n,  x  =  9  1.000000 
 n,  x  =  10  2.000000 
 n,  x  =  11  3.000000 
 n,  x  =  12  4.000000 
 n,  x  =  13  5.000000 
 n,  x  =  14  6.000000 
 n,  x  =  15  7.000000 


In [ ]:
!nvcc prac1b.cu -o prac1b -I/content/drive/MyDrive/CHPS0802


In [ ]:
!./prac1b

Thread ID: 8
Thread ID: 9
Thread ID: 10
Thread ID: 11
Thread ID: 12
Thread ID: 13
Thread ID: 14
Thread ID: 15
Thread ID: 0
Thread ID: 1
Thread ID: 2
Thread ID: 3
Thread ID: 4
Thread ID: 5
Thread ID: 6
Thread ID: 7
 n,  x  =  0  0.000000 
 n,  x  =  1  1.000000 
 n,  x  =  2  2.000000 
 n,  x  =  3  3.000000 
 n,  x  =  4  4.000000 
 n,  x  =  5  5.000000 
 n,  x  =  6  6.000000 
 n,  x  =  7  7.000000 
 n,  x  =  8  0.000000 
 n,  x  =  9  1.000000 
 n,  x  =  10  2.000000 
 n,  x  =  11  3.000000 
 n,  x  =  12  4.000000 
 n,  x  =  13  5.000000 
 n,  x  =  14  6.000000 
 n,  x  =  15  7.000000 
Index 0: 0.000000 + 0.000000 = 0.000000
Index 1: 1.000000 + 2.000000 = 3.000000
Index 2: 2.000000 + 4.000000 = 6.000000
Index 3: 3.000000 + 6.000000 = 9.000000
Index 4: 4.000000 + 8.000000 = 12.000000
Index 5: 5.000000 + 10.000000 = 15.000000
Index 6: 6.000000 + 12.000000 = 18.000000
Index 7: 7.000000 + 14.000000 = 21.000000
Index 8: 8.000000 + 16.000000 = 24.000000
Index 9: 9.000000 + 18.0000

In [ ]:
!diff prac1a.cu prac1b.cu


9a10,12
> #include <helper_cuda.h>
> 
> 
17c20,21
< 
---
>   // Print the thread ID
>   printf("Thread ID: %d\n", tid);
26c30
< int main(int argc, char **argv)
---
> int main(int argc, const char **argv)
30a35,38
>   // initialise card
> 
>   findCudaDevice(argc, argv);
> 
40c48
<   cudaMalloc((void **)&d_x, nsize*sizeof(float));
---
>   checkCudaErrors(cudaMalloc((void **)&d_x, nsize*sizeof(float)));
43c51
< 
---
>   
44a53
>   getLastCudaError("my_first_kernel execution failed\n");
48c57,58
<   cudaMemcpy(h_x,d_x,nsize*sizeof(float),cudaMemcpyDeviceToHost);
---
>   checkCudaErrors( cudaMemcpy(h_x,d_x,nsize*sizeof(float),
>                  cudaMemcpyDeviceToHost) );
54c64
<   cudaFree(d_x);
---
>   checkCudaErrors(cudaFree(d_x));


In [1]:
%%writefile util.h

#ifndef UTILS_H
#define UTILS_H

int Add(int, int);

#endif //UTILS_H

Writing util.h


In [2]:
%%writefile util.cpp

int Add(int a, int b) {
    return a + b;
}

Writing util.cpp


In [3]:
%%writefile main.cpp

#include <iostream>
#include "util.h"

int main() {
    std::cout << Add(1, 2) << std::endl;
    return EXIT_SUCCESS;
}

Writing main.cpp


In [4]:
%%writefile test.cpp
#include <gtest/gtest.h>
#include "util.h"

TEST(AdditionTest, HandlesPositiveInput) {
    EXPECT_EQ(Add(1, 2), 3);
}

TEST(AdditionTest, HandlesNegativeInput) {
    EXPECT_EQ(Add(-1, -2), -3);
}

int main(int argc, char **argv) {
    ::testing::InitGoogleTest(&argc, argv);
    return RUN_ALL_TESTS();
}

Writing test.cpp


In [5]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.30)
project(google_test)

set(CMAKE_CXX_STANDARD 14)

add_library(util_lib util.cpp)

add_executable(addition main.cpp)

target_link_libraries(addition util_lib)

################
#Test unitaires
###############
include(FetchContent)

FetchContent_Declare(
        googletest
        # Specify the commit you depend on and update it regularly.
        URL https://github.com/google/googletest/archive/5376968f6948923e2411081fd9372e71a59d8e77.zip
)
# For Windows: Prevent overriding the parent project's compiler/linker settings
set(gtest_force_shared_crt ON CACHE BOOL "" FORCE)
FetchContent_MakeAvailable(googletest)

add_executable(runUnitTests test.cpp)

target_link_libraries(runUnitTests gtest gtest_main)

target_link_libraries(runUnitTests util_lib)

add_test(NAME example_test COMMAND runUnitTests)


Writing CMakeLists.txt


In [6]:
%%bash
mkdir build
cd build
cmake ..
make

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- [download 1% complete]
-- [download 3% complete]
-- [download 4% complete]
-- [download 6% complete]
-- [download 7% complete]
-- [download 8% complete]
-- [download 10% complete]
-- [download 11% complete]
-- [download 13% complete]
-- [download 14% complete]
-- [download 15% complete]
-- [download 16% complete]
-- [download 18% complete]
-- [download 19% complete]
-- [download 21% complete]
-- [download 22% complete]
-- [download 24% complete]
-- [download 25% complete]
-- [download 27

CMake Deprecation Warning at build/_deps/googletest-src/CMakeLists.txt:4 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Deprecation Warning at build/_deps/googletest-src/googlemock/CMakeLists.txt:39 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


CMake Deprecation Warning at build/_deps/googletest-src/googletest/CMakeLists.txt:49 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

In [7]:
%%bash
cd build/
./runUnitTests

[==========] Running 2 tests from 1 test suite.
[----------] Global test environment set-up.
[----------] 2 tests from AdditionTest
[ RUN      ] AdditionTest.HandlesPositiveInput
[       OK ] AdditionTest.HandlesPositiveInput (0 ms)
[ RUN      ] AdditionTest.HandlesNegativeInput
[       OK ] AdditionTest.HandlesNegativeInput (0 ms)
[----------] 2 tests from AdditionTest (0 ms total)

[----------] Global test environment tear-down
[==========] 2 tests from 1 test suite ran. (0 ms total)
[  PASSED  ] 2 tests.
